# OpenAI Whisper API + pyannote Pipeline

## **Pipeline Overview**
This notebook creates a **cloud-based transcription pipeline** using OpenAI's Whisper API combined with local speaker diarization. We're testing this approach against our existing local models to evaluate:

### **Performance Comparison**
- **Speed**: API processing vs local GPU/CPU intensive models
- **Accuracy**: Latest OpenAI model vs open-source alternatives  
- **Resource Usage**: Cloud processing vs local memory/compute requirements
- **Cost**: API fees vs electricity/hardware costs

### **Expected Advantages**
- **Faster processing** - Cloud infrastructure vs local hardware
- **Latest model** - Most recent Whisper improvements not yet in open-source
- **No local resources** - No GPU memory limitations or long processing times
- **Consistent results** - Same performance regardless of local hardware

### **Pipeline Architecture**
```
Audio Input → OpenAI Whisper API → Transcript with Timestamps
     ↓
Speaker Diarization (pyannote) → Speaker-labeled Segments
     ↓
Integration Layer → Combined Transcript + Speaker IDs
     ↓
Analysis Ready → Emotion Detection, Bias Analysis, Political Insights
```

### **What We'll Build Next**
1. **Transcription Quality Comparison** - OpenAI vs WhisperX vs Local models
2. **Speed Benchmarking** - Processing time across different approaches
3. **Accuracy Metrics** - Word error rates and speaker identification precision
4. **Cost Analysis** - API costs vs computational expenses
5. **Integration with Emotion Analysis** - Seamless pipeline to bias detection
6. **Political Analysis Dashboard** - Speaker patterns, sentiment trends, bias indicators

This pipeline will serve as our **premium accuracy option** for critical analysis while maintaining our local alternatives for cost-sensitive or private processing needs.

---

In [1]:
# ============================================================
# SETUP - Import libraries and initialize components
# ============================================================

# Core libraries
import os
import json
import pandas as pd
import numpy as np
from datetime import datetime
import time

# OpenAI API setup
from dotenv import load_dotenv
from openai import OpenAI

# Display options for better pandas output
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Load environment variables from .env file
load_dotenv()

# Initialize OpenAI client with API key
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Define audio file paths 
WAV_FILE = "../data/US_DebateAudio.wav"      # For pyannote diarization
MP3_FILE = "../data/US_DebateAudio.mp3"  # For Whisper API transcription

# Create outputs directory if it doesn't exist
os.makedirs("./outputs", exist_ok=True)

# Verify setup
api_key_loaded = bool(os.getenv("OPENAI_API_KEY"))
wav_exists = os.path.exists(WAV_FILE)
mp3_exists = os.path.exists(MP3_FILE)

print("=== SETUP COMPLETE ===")
print(f"API key loaded: {api_key_loaded}")
print(f"WAV file exists: {wav_exists} ({WAV_FILE})")
print(f"MP3 file exists: {mp3_exists} ({MP3_FILE})")
print(f"Outputs directory ready: {os.path.exists('./outputs')}")

if not all([api_key_loaded, wav_exists, mp3_exists]):
    print("Some required components are missing!")
else:
    print("Ready to proceed with pipeline!")

=== SETUP COMPLETE ===
API key loaded: True
WAV file exists: True (../data/US_DebateAudio.wav)
MP3 file exists: True (../data/US_DebateAudio.mp3)
Outputs directory ready: True
Ready to proceed with pipeline!


## Step 2 – Transcription with OpenAI Whisper API

We'll use OpenAI's Whisper API to transcribe our MP3 file and get **segments with timestamps**.

### **What this does:**
- Sends audio to OpenAI's cloud-based Whisper model
- Gets back transcript text + individual segments with start/end times
- Each segment is a phrase or sentence with precise timing
- This gives us the **"what was said"** part of our pipeline

### **API Settings:**
- **Model**: `whisper-1` (latest available via API)
- **Format**: `verbose_json` (includes timing and metadata)
- **Granularity**: `segment` level (phrases, not just words)

In [2]:
# ============================================================
# STEP 2: Transcribe audio with OpenAI Whisper API
# ============================================================

def transcribe_with_openai(audio_path):
    """
    Transcribe audio using OpenAI Whisper API
    
    Args:
        audio_path (str): Path to the audio file (MP3/WAV/etc.)
    
    Returns:
        OpenAI transcription response object with segments and text
    """
    print(f"Opening audio file: {audio_path}")
    
    # Check file size (OpenAI has 25MB limit)
    file_size_mb = os.path.getsize(audio_path) / (1024 * 1024)
    print(f"File size: {file_size_mb:.2f} MB")
    
    if file_size_mb > 25:
        print("WARNING: File exceeds 25MB limit - API call will fail")
        return None
    
    print("Sending to OpenAI Whisper API...")
    start_time = time.time()
    
    try:
        with open(audio_path, "rb") as audio_file:
            # Call OpenAI Whisper API with detailed response format
            response = client.audio.transcriptions.create(
                model="whisper-1",                          # Latest Whisper model
                file=audio_file,                           # Audio file object
                response_format="verbose_json",            # Get detailed response with metadata
                timestamp_granularities=["segment"]       # Get segment-level timestamps
            )
        
        elapsed_time = time.time() - start_time
        print(f"Transcription completed in {elapsed_time:.2f} seconds")
        
        return response
    
    except Exception as e:
        print(f"API Error: {e}")
        return None

# Run transcription on our MP3 file
print("Starting transcription with OpenAI Whisper API...")
whisper_result = transcribe_with_openai(MP3_FILE)

# Display results
if whisper_result:
    print("\n" + "="*60)
    print("TRANSCRIPTION RESULTS")
    print("="*60)
    
    # Basic info about the transcription
    print(f"Language detected: {getattr(whisper_result, 'language', 'unknown')}")
    print(f"Total duration: {getattr(whisper_result, 'duration', 0):.2f} seconds")
    print(f"Full text length: {len(whisper_result.text)} characters")
    
    # Show number of segments if available
    if hasattr(whisper_result, 'segments') and whisper_result.segments:
        print(f"Number of segments: {len(whisper_result.segments)}")
    else:
        print("No segment data returned")
    
    # Preview first 200 characters of transcript
    print(f"\nTranscript preview (first 200 chars):")
    print("-" * 40)
    preview_text = whisper_result.text[:200] + "..." if len(whisper_result.text) > 200 else whisper_result.text
    print(preview_text)
    
else:
    print("Transcription failed - cannot proceed to next step")

Starting transcription with OpenAI Whisper API...
Opening audio file: ../data/US_DebateAudio.mp3
File size: 10.87 MB
Sending to OpenAI Whisper API...
Transcription completed in 61.27 seconds

TRANSCRIPTION RESULTS
Language detected: english
Total duration: 565.77 seconds
Full text length: 8463 characters
Number of segments: 209

Transcript preview (first 200 chars):
----------------------------------------
She doesn't have a plan. She copied Biden's plan, and it's like four sentences, like run, spot, run, four sentences that are just, oh, we'll try and lower taxes. She doesn't have a plan. Take a look a...


## Step 3 – Convert Whisper segments to DataFrame

Now we'll extract the segment data from the OpenAI response and put it into a clean pandas DataFrame.

### **What this step does:**
- Extracts individual segments from `whisper_result.segments`
- Each segment has: start time, end time, and text content
- Creates a structured DataFrame we can easily work with
- This gives us the foundation for merging with speaker data

### **DataFrame structure:**
- `start_s`: When the segment starts (seconds)
- `end_s`: When the segment ends (seconds) 
- `text`: What was said in that time period

In [4]:
# ============================================================
# STEP 3: Convert Whisper segments to pandas DataFrame
# ============================================================

def extract_whisper_segments(whisper_response):
    """
    Extract segment data from OpenAI Whisper API response
    
    Args:
        whisper_response: Response object from OpenAI API
        
    Returns:
        pandas.DataFrame with columns: start_s, end_s, text
    """
    segments_list = []
    
    # Check if we have segments in the response
    if hasattr(whisper_response, 'segments') and whisper_response.segments:
        print(f"Processing {len(whisper_response.segments)} segments...")
        
        for i, segment in enumerate(whisper_response.segments):
            # Extract the key information from each segment
            segment_data = {
                'start_s': round(segment.start, 2),      # Start time in seconds
                'end_s': round(segment.end, 2),          # End time in seconds
                'text': segment.text.strip()             # Clean up the text
            }
            segments_list.append(segment_data)
            
        print(f"Extracted {len(segments_list)} segments successfully")
    else:
        print("o segments found in response")
    
    # Create DataFrame from the list of segment dictionaries
    df = pd.DataFrame(segments_list)
    return df

# Only proceed if we have a successful transcription
if whisper_result:
    print("Converting Whisper segments to DataFrame...")
    
    # Extract segments into a clean DataFrame
    whisper_df = extract_whisper_segments(whisper_result)
    
    # Display information about our DataFrame
    print(f"\n{'='*50}")
    print("WHISPER SEGMENTS DATAFRAME")
    print(f"{'='*50}")
    
    print(f"DataFrame shape: {whisper_df.shape}")
    print(f"Columns: {list(whisper_df.columns)}")
    
    if len(whisper_df) > 0:
        print(f"Time range: {whisper_df['start_s'].min():.1f}s to {whisper_df['end_s'].max():.1f}s")
        print(f"Total text length: {whisper_df['text'].str.len().sum()} characters")
        
        print(f"\nFirst 5 segments:")
        print(whisper_df.head())
        
        print(f"\nDataFrame info:")
        print(whisper_df.info())
    else:
        print("No segments to display")
        
else:
    print("Cannot proceed - no whisper_result from previous step")

Converting Whisper segments to DataFrame...
Processing 209 segments...
Extracted 209 segments successfully

WHISPER SEGMENTS DATAFRAME
DataFrame shape: (209, 3)
Columns: ['start_s', 'end_s', 'text']
Time range: 0.0s to 557.6s
Total text length: 8255 characters

First 5 segments:
   start_s  end_s                                               text
0     0.00   2.00                           She doesn't have a plan.
1     2.00   6.72  She copied Biden's plan, and it's like four se...
2     6.72  10.60  like run, spot, run, four sentences that are j...
3    10.60  12.76                     oh, we'll try and lower taxes.
4    12.76  13.68                           She doesn't have a plan.

DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 209 entries, 0 to 208
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   start_s  209 non-null    float64
 1   end_s    209 non-null    float64
 2   text     209 non-null    obj

## Step 4 – Speaker Diarization with pyannote

Now we'll use pyannote to identify **who spoke when** in our audio file.

### **What this step does:**
- Uses pyannote's pre-trained speaker diarization model
- Analyzes the WAV file to detect different speakers
- Creates time segments showing when each speaker was talking
- Gives us the **"who said it"** part of our pipeline

### **pyannote Setup:**
- **Model**: `pyannote/speaker-diarization-3.1` (latest version)
- **Input**: WAV file (better quality for speaker detection)
- **Output**: Speaker segments with start/end times and speaker IDs


In [ ]:
# ============================================================
# STEP 4: Install and setup pyannote
# ============================================================

# Install pyannote if not already installed
try:
    import torch
    from pyannote.audio import Pipeline
    print("pyannote.audio already installed")
except ImportError:
    print("Installing pyannote.audio...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyannote.audio"])
    
    # Import after installation
    import torch
    from pyannote.audio import Pipeline
    print("pyannote.audio installed successfully")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

c:\Users\norak\SpeakSense\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\norak\SpeakSense\venv\Lib\site-packages\pyannote\audio\core\io.py:212: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  torchaudio.list_audio_backends()


pyannote.audio already installed
PyTorch version: 2.8.0+cpu
CUDA available: False


In [ ]:
# ============================================================
# STEP 4: Run speaker diarization with pyannote
# ============================================================

def run_diarization(audio_path, hf_token=None):
    """
    Run speaker diarization on audio file
    
    Args:
        audio_path (str): Path to WAV audio file
        hf_token (str, optional): Hugging Face token for model access
        
    Returns:
        pandas.DataFrame with columns: start_s, end_s, speaker
    """
    print(f"Running diarization on: {audio_path}")
    
    try:
        # Load the diarization pipeline
        if hf_token:
            pipeline = Pipeline.from_pretrained(
                "pyannote/speaker-diarization-3.1",
                use_auth_token=hf_token
            )
        else:
            # Try without token first
            pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1")
        
        print("Diarization model loaded successfully")
        
        # Run diarization
        print("Processing audio for speaker diarization...")
        start_time = time.time()
        
        diarization = pipeline(audio_path)
        
        elapsed_time = time.time() - start_time
        print(f"Diarization completed in {elapsed_time:.2f} seconds")
        
        # Convert to DataFrame
        segments_list = []
        for turn, _, speaker in diarization.itertracks(yield_label=True):
            segments_list.append({
                'start_s': round(turn.start, 2),
                'end_s': round(turn.end, 2),
                'speaker': speaker
            })
        
        diarize_df = pd.DataFrame(segments_list)
        print(f"Created diarization DataFrame with {len(diarize_df)} segments")
        
        return diarize_df
        
    except Exception as e:
        print(f"Diarization failed: {e}")
        return None

# Run diarization
print("Starting speaker diarization with pyannote...")

# Try to get HF token from environment (optional)
hf_token = os.getenv("HUGGINGFACE_TOKEN")

# Run diarization on our WAV file
diarize_df = run_diarization(WAV_FILE, hf_token)

# Display results
if diarize_df is not None and len(diarize_df) > 0:
    print("\n" + "="*60)
    print("SPEAKER DIARIZATION RESULTS")
    print("="*60)
    
    # Basic stats
    unique_speakers = diarize_df['speaker'].nunique()
    total_segments = len(diarize_df)
    
    print(f"Unique speakers detected: {unique_speakers}")
    print(f"Total speaker segments: {total_segments}")
    print(f"Time range: {diarize_df['start_s'].min():.1f}s to {diarize_df['end_s'].max():.1f}s")
    
    # Show speaker distribution
    print(f"\nSpeaker segment counts:")
    speaker_counts = diarize_df['speaker'].value_counts()
    for speaker, count in speaker_counts.items():
        print(f"   {speaker}: {count} segments")
    
    # Show first few segments
    print(f"\nFirst 10 diarization segments:")
    print(diarize_df.head(10))
    
    print(f"\nDataFrame info:")
    print(diarize_df.info())
    


Starting speaker diarization with pyannote...
Running diarization on: ../data/US_DebateAudio.wav


c:\Users\norak\SpeakSense\venv\Lib\site-packages\speechbrain\utils\torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()
C:\Python312\Lib\inspect.py:1001: UserWarning: Module 'speechbrain.pretrained' was deprecated, redirecting to 'speechbrain.inference'. Please update your script. This is a change from SpeechBrain 1.0. See: https://github.com/speechbrain/speechbrain/releases/tag/v1.0.0
  if ismodule(module) and hasattr(module, '__file__'):
c:\Users\norak\SpeakSense\venv\Lib\site-packages\pyannote\audio\core\io.py:212: UserWarning: torchaudio._backen

Diarization model loaded successfully
Processing audio for speaker diarization...


c:\Users\norak\SpeakSense\venv\Lib\site-packages\torchaudio\_backend\utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
c:\Users\norak\SpeakSense\venv\Lib\site-packages\pyannote\audio\models\blocks\pooling.py:104: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1839.)
  std = sequences.std(dim=-1, correction=1)
c:\Users\norak\SpeakSense\venv\Lib\site-packages\pyannote\audio\c

✅ Diarization completed in 504.36 seconds
Created diarization DataFrame with 54 segments

SPEAKER DIARIZATION RESULTS
Unique speakers detected: 3
Total speaker segments: 54
Time range: 0.0s to 557.2s

Speaker segment counts:
   SPEAKER_02: 34 segments
   SPEAKER_00: 12 segments
   SPEAKER_01: 8 segments

First 10 diarization segments:
   start_s   end_s     speaker
0     0.03   15.99  SPEAKER_00
1    16.10   28.38  SPEAKER_02
2    28.52   38.91  SPEAKER_02
3    39.37   53.93  SPEAKER_02
4    54.08   62.35  SPEAKER_02
5    62.33   66.03  SPEAKER_00
6    66.50   70.11  SPEAKER_00
7    70.30  168.97  SPEAKER_00
8   169.08  179.67  SPEAKER_00
9   179.73  184.49  SPEAKER_02

DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   start_s  54 non-null     float64
 1   end_s    54 non-null     float64
 2   speaker  54 non-null     object 
dtypes: float

In [12]:
# ============================================================
# STEP 5: Merge OpenAI Whisper + pyannote for Speaker-Labeled Transcript
# ============================================================

def merge_transcript_with_speakers(whisper_df, diarize_df):
    """
    Merge OpenAI Whisper transcript segments with pyannote speaker diarization
    
    Args:
        whisper_df: DataFrame with columns [start_s, end_s, text]
        diarize_df: DataFrame with columns [start_s, end_s, speaker]
        
    Returns:
        DataFrame with columns [start_s, end_s, text, speaker]
    """
    print("Merging Whisper transcript with speaker diarization...")
    
    merged_segments = []
    
    for _, whisper_seg in whisper_df.iterrows():
        w_start = whisper_seg['start_s']
        w_end = whisper_seg['end_s']
        w_text = whisper_seg['text']
        
        # Find overlapping speaker segments
        overlaps = []
        
        for _, speaker_seg in diarize_df.iterrows():
            s_start = speaker_seg['start_s']
            s_end = speaker_seg['end_s']
            speaker = speaker_seg['speaker']
            
            # Calculate overlap between whisper segment and speaker segment
            overlap_start = max(w_start, s_start)
            overlap_end = min(w_end, s_end)
            
            if overlap_start < overlap_end:  # There is overlap
                overlap_duration = overlap_end - overlap_start
                overlaps.append({
                    'speaker': speaker,
                    'overlap_duration': overlap_duration,
                    'overlap_ratio': overlap_duration / (w_end - w_start)
                })
        
        # Assign speaker based on maximum overlap
        if overlaps:
            # Sort by overlap duration and pick the speaker with most overlap
            best_match = max(overlaps, key=lambda x: x['overlap_duration'])
            assigned_speaker = best_match['speaker']
        else:
            assigned_speaker = 'UNKNOWN'
        
        merged_segments.append({
            'start_s': w_start,
            'end_s': w_end,
            'text': w_text,
            'speaker': assigned_speaker
        })
    
    merged_df = pd.DataFrame(merged_segments)
    return merged_df

# Only proceed if we have both whisper and diarization results
if ('whisper_df' in locals() and whisper_df is not None and len(whisper_df) > 0 and
    'diarize_df' in locals() and diarize_df is not None and len(diarize_df) > 0):
    
    print("Merging OpenAI Whisper transcript with pyannote speakers...")
    
    # Merge the data
    merged_df = merge_transcript_with_speakers(whisper_df, diarize_df)
    
    # Display results
    print("\n" + "="*60)
    print("MERGED TRANSCRIPT WITH SPEAKERS")
    print("="*60)
    
    print(f"Merge Results:")
    print(f"   Total segments: {len(merged_df)}")
    print(f"   Unique speakers: {merged_df['speaker'].nunique()}")
    print(f"   Time range: {merged_df['start_s'].min():.1f}s to {merged_df['end_s'].max():.1f}s")
    
    # Speaker distribution
    print(f"\nSpeaker Distribution:")
    speaker_stats = merged_df['speaker'].value_counts()
    for speaker, count in speaker_stats.items():
        total_words = merged_df[merged_df['speaker'] == speaker]['text'].str.split().str.len().sum()
        total_time = (merged_df[merged_df['speaker'] == speaker]['end_s'] - 
                     merged_df[merged_df['speaker'] == speaker]['start_s']).sum()
        
        print(f"   {speaker}: {count} segments, {total_words} words, {total_time:.1f}s speaking time")
    
    # Show sample of merged data
    print(f"\nSample Merged Segments:")
    for i, row in merged_df.head(10).iterrows():
        print(f"   [{row['start_s']:6.1f}s - {row['end_s']:6.1f}s] {row['speaker']}: {row['text'][:80]}...")
    
    # Create full transcript by speaker
    print(f"\n Full Transcript by Speaker:")
    print("-" * 50)
    
    for speaker in merged_df['speaker'].unique():
        if speaker != 'UNKNOWN':  # Skip unknown speakers for cleaner output
            speaker_text = merged_df[merged_df['speaker'] == speaker]['text'].str.cat(sep=' ')
            word_count = len(speaker_text.split())
            
            print(f"\n🎤 {speaker} ({word_count} words):")
            print(f"{speaker_text[:300]}..." if len(speaker_text) > 300 else speaker_text)
    
    # Save merged results
    output_file = "../outputs/openai_whisper_with_speakers.csv"
    merged_df.to_csv(output_file, index=False)
    
    
    
else:
    missing = []
    if 'whisper_df' not in locals() or whisper_df is None or len(whisper_df) == 0:
        missing.append("OpenAI Whisper transcript (whisper_df)")
    if 'diarize_df' not in locals() or diarize_df is None or len(diarize_df) == 0:
        missing.append("pyannote diarization (diarize_df)")
    
    print("Cannot merge - missing data:")
    for item in missing:
        print(f"   - {item}")
    print("Run the previous steps to generate both datasets first")

Merging OpenAI Whisper transcript with pyannote speakers...
Merging Whisper transcript with speaker diarization...

MERGED TRANSCRIPT WITH SPEAKERS
Merge Results:
   Total segments: 209
   Unique speakers: 3
   Time range: 0.0s to 557.6s

Speaker Distribution:
   SPEAKER_02: 107 segments, 774 words, 304.8s speaking time
   SPEAKER_00: 100 segments, 723 words, 246.8s speaking time
   SPEAKER_01: 2 segments, 13 words, 4.5s speaking time

Sample Merged Segments:
   [   0.0s -    2.0s] SPEAKER_00: She doesn't have a plan....
   [   2.0s -    6.7s] SPEAKER_00: She copied Biden's plan, and it's like four sentences,...
   [   6.7s -   10.6s] SPEAKER_00: like run, spot, run, four sentences that are just,...
   [  10.6s -   12.8s] SPEAKER_00: oh, we'll try and lower taxes....
   [  12.8s -   13.7s] SPEAKER_00: She doesn't have a plan....
   [  13.7s -   14.6s] SPEAKER_00: Take a look at her plan....
   [  14.6s -   16.0s] SPEAKER_00: She doesn't have a plan....
   [  16.0s -   19.2s] SPEAKER_02